In [13]:
import sqlite3
import pandas as pd

# Crear o conectar con la base de datos
conexion = sqlite3.connect("cervezas.db")

# Crear cursor
cursor = conexion.cursor()

In [14]:
cursor.executescript("""
CREATE TABLE IF NOT EXISTS CERVEZAS (
    CodC TEXT PRIMARY KEY,
    Envase TEXT,
    Capacidad REAL,
    Stock INTEGER
);

CREATE TABLE IF NOT EXISTS BARES (
    CodB TEXT PRIMARY KEY,
    Cif TEXT,
    Nombre TEXT,
    Localidad TEXT
);

CREATE TABLE IF NOT EXISTS EMPLEADOS (
    CodE INTEGER PRIMARY KEY,
    Nombre TEXT,
    Sueldo REAL
);

CREATE TABLE IF NOT EXISTS REPARTO (
    CodE INTEGER,
    CodB TEXT,
    CodC TEXT,
    Fecha TEXT,
    Cantidad INTEGER,
    FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
    FOREIGN KEY (CodB) REFERENCES BARES(CodB),
    FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
""")

conexion.commit()

In [15]:
cursor.executemany("""
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock)
VALUES (?, ?, ?, ?)
""", [
    ("01", "Botella", 0.2, 3600),
    ("02", "Botella", 0.33, 1200),
    ("03", "Lata", 0.33, 2400),
    ("04", "Botella", 1, 288),
    ("05", "Barril", 60, 30)
])

In [16]:
cursor.executemany("""
INSERT INTO BARES (CodB, Cif, Nombre, Localidad)
VALUES (?, ?, ?, ?)
""", [
    ("001", "11111111X", "Stop", "Villa Botijo"),
    ("002", "22222222Y", "Las Vegas", "Villa Botijo"),
    ("003", "-", "Club Social", "Las Ranas"),
    ("004", "33333333Z", "Otra Ronda", "La Esponja")
])

In [17]:
cursor.executemany("""
INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo)
VALUES (?, ?, ?)
""", [
    (1, "Prudencio Caminero", 120000),
    (2, "Vicente Merario", 110000),
    (3, "Valentin Siempre", 100000)
])

In [18]:
cursor.executemany("""
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, "001", "01", "2005-10-21", 240),
    (1, "001", "02", "2005-10-21", 48),
    (1, "002", "03", "2005-10-22", 60),
    (1, "004", "05", "2005-10-22", 4),
    (2, "002", "03", "2005-10-22", 48),
    (2, "002", "05", "2005-10-23", 2),
    (2, "004", "01", "2005-10-23", 480),
    (2, "004", "02", "2005-10-24", 72),
    (3, "003", "03", "2005-10-24", 48),
    (3, "003", "04", "2005-10-25", 20)
])

In [19]:
conexion.commit()

In [20]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    cursor.execute(query)

    # Almacena los datos de la query 
    ans = cursor.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in cursor.description]

    return pd.DataFrame(ans,columns=names)

1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.

In [22]:
query = '''
SELECT DISTINCT empleados.Nombre
FROM EMPLEADOS
INNER JOIN REPARTO
ON empleados.CodE = reparto.CodE
INNER JOIN BARES
ON reparto.CodB = bares.CodB
WHERE bares.Nombre = "Stop" AND reparto.Fecha BETWEEN "2005-10-17" AND "2005-10-23";
'''

df = sql_query(query)
df

,Nombre
0,Prudencio Caminero


2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.

In [23]:
query = '''
SELECT DISTINCT bares.Cif, bares.Nombre
FROM bares
INNER JOIN REPARTO
ON bares.Codb = reparto.Codb
INNER JOIN cervezas
ON reparto.Codc = cervezas.Codc
WHERE cervezas.envase = "Botella" AND cervezas.capacidad < 1
ORDER BY bares.Localidad;
'''

df = sql_query(query)
df

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.

In [24]:
query = '''
SELECT DISTINCT bares.Nombre, cervezas.envase, cervezas.capacidad, reparto.Fecha, reparto.cantidad
FROM empleados
INNER JOIN REPARTO
ON empleados.Code = reparto.Code
INNER JOIN cervezas
ON reparto.Codc = cervezas.Codc
INNER JOIN bares
ON reparto.Codb = bares.Codb
WHERE empleados.Nombre = "Prudencio Caminero";
'''

df = sql_query(query)
df

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.

In [25]:
query = '''
SELECT DISTINCT bares.Nombre
FROM bares
INNER JOIN REPARTO
ON bares.Codb = reparto.Codb
INNER JOIN cervezas
ON reparto.Codc = cervezas.Codc
WHERE cervezas.envase = "Botella" AND cervezas.Capacidad IN (0.2,0.33);
'''

df = sql_query(query)
df

,Nombre
0,Stop
1,Otra Ronda


5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella. 

In [26]:
query = '''
SELECT empleados.Nombre
FROM empleados
INNER JOIN REPARTO
ON empleados.Code = reparto.Code
INNER JOIN cervezas
ON reparto.Codc = cervezas.Codc
INNER JOIN bares
ON reparto.Codb = bares.Codb
WHERE cervezas.envase = "Botella" AND bares.Nombre = "Stop" and bares.Nombre = "Las Vegas";
'''

df = sql_query(query)
df

,Nombre


6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.

In [27]:
query = '''
SELECT empleados.Nombre, count(reparto.fecha)
FROM empleados
INNER JOIN REPARTO
ON empleados.Code = reparto.Code
INNER JOIN bares
ON reparto.Codb = bares.Codb
WHERE bares.localidad != "Villa Botijo"
group by reparto.fecha;
'''

df = sql_query(query)
df

,Nombre,count(reparto.fecha)
0,Prudencio Caminero,1
1,Vicente Merario,1
2,Vicente Merario,2
3,Valentin Siempre,1


7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

In [28]:
query = '''
SELECT bares.Nombre, bares.localidad, sum(reparto.Cantidad * cervezas.Capacidad) as cantidad
FROM bares
INNER JOIN REPARTO
ON bares.Codb = reparto.Codb
INNER JOIN cervezas
ON reparto.Codc = cervezas.Codc
group by cantidad
order by cantidad desc
limit 1;
'''

df = sql_query(query)
df

,Nombre,Localidad,cantidad
0,Otra Ronda,La Esponja,240.0


8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.

In [29]:
query = '''
SELECT bares.CodB, bares.Nombre, bares.Localidad
FROM bares
JOIN REPARTO
ON bares.CodB = reparto.CodB
JOIN CERVEZAS
ON reparto.CodC = cervezas.CodC
WHERE cervezas.Envase = "Botella" AND cervezas.Capacidad < 1;
'''

df = sql_query(query)
df

,CodB,Nombre,Localidad
0,001,Stop,Villa Botijo
1,001,Stop,Villa Botijo
2,004,Otra Ronda,La Esponja
3,004,Otra Ronda,La Esponja


9. Subir un 5% el sueldo del empleado que más días haya trabajado.

In [30]:
query = '''
SELECT empleados.Nombre, empleados.Sueldo, count(DISTINCT reparto.Fecha) as dias_trabajados
FROM empleados
JOIN REPARTO
ON empleados.Code = reparto.Code
GROUP BY empleados.Nombre, empleados.Sueldo
ORDER BY dias_trabajados desc
LIMIT 1;
'''

df = sql_query(query)
df

,Nombre,Sueldo,dias_trabajados
0,Vicente Merario,110000.0,3


In [34]:
query = '''
UPDATE Empleados
SET Sueldo = Sueldo * 1.05
WHERE Nombre = 'Vicente Merario';
'''
cursor.execute(query);

10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [36]:
query = '''
INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad)
VALUES ((SELECT CodE FROM Empleados WHERE Empleados.Nombre = "Vicente Merario"),
(SELECT CodB FROM Bares WHERE Bares.Nombre = "Stop"),
(SELECT CodC FROM Cervezas WHERE Cervezas.Envase = "Lata"), "2005-10-26", 48)
'''

cursor.execute(query);